# ClusterAPI (Topology-Cluster)

Cluster API ist ein Kubernetes-Projekt zur deklarativen Erstellung und Verwaltung von Kubernetes-Clustern. Es bildet den gesamten Cluster-Lifecycle über Kubernetes-Ressourcen und Controller ab.

Die Ressourcen sind hierarchisch aufgebaut: Ein `Cluster` beschreibt den logischen Kubernetes-Cluster. Die Control Plane wird über die Ressource `KubeadmControlPlane` verwaltet, während Worker Nodes über `MachineDeployments`, darunterliegende `MachineSets` und einzelne `Machines` abgebildet werden. Jede `Machine` repräsentiert dabei eine konkrete virtuelle oder physische Instanz, die später als Kubernetes Node registriert wird.

```text
Cluster
├── spec.topology
│   └── definiert Control Plane und Worker-Klassen
├── KubeadmControlPlane
│   └── Machines
└── MachineDeployments
    └── MachineSets
        └── Machines
            └── Kubernetes Nodes
```

Ein `Cluster` beschreibt den logischen Kubernetes-Cluster. Bei einem Topology-Cluster definiert `Cluster.spec.topology` den gewünschten Aufbau. Der Topology-Controller erzeugt und verwaltet daraus die `KubeadmControlPlane` sowie die `MachineDeployments`; darunter folgen weiterhin `MachineSets`, `Machines` und die registrierten Kubernetes Nodes.

Im Gegensatz zu Terraform prüft Cluster API laufend, ob der tatsächliche Zustand dem gewünschten Zustand entspricht, und korrigiert Abweichungen automatisch. Es ermöglicht einheitliche Cluster-Vorlagen, kontrollierte Updates, das Skalieren von Nodes und den automatischen Ersatz defekter Maschinen.

Da Cluster, Control Plane und Worker Nodes als Kubernetes-Ressourcen verwaltet werden, lassen sie sich direkt mit GitOps, Berechtigungen, Richtlinien und Monitoring verbinden. 

**Der grösste Vorteil liegt deshalb nicht nur in der Erstellung von Clustern, sondern vor allem in deren einheitlichem und automatisiertem Betrieb.**

---

Installation ClusterAPI CLI

In [ ]:
%%bash
ARCH_RAW=$(uname -m)
case "$ARCH_RAW" in
  x86_64)
    ARCH="amd64"
    ;;
  aarch64|arm64)
    ARCH="arm64"
    ;;
esac

echo "✅ [INFO] Erkannte Architektur: $ARCH_RAW → $ARCH"

curl -L https://github.com/kubernetes-sigs/cluster-api/releases/download/v1.13.3/clusterctl-linux-${ARCH} -o clusterctl
sudo install -o root -g root -m 0755 clusterctl /usr/local/bin/clusterctl

### Kind Cluster

Erstellen mit Zugriff auf docker.sock


In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kind create cluster  --name capi-kind --config=- <<EOF
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
name: kind
nodes:
- role: control-plane
  extraMounts:
  - hostPath: /var/run/docker.sock
    containerPath: /var/run/docker.sock
EOF

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl cluster-info

### ClusterAPI Initialisieren

Initialisieren des Management-Cluster im Kind Cluster. 

Dabei werden die benötigten Cluster-API-Komponenten sowie die Controller und Custom Resources für den Kind Cluster installiert. Der Management-Cluster kann anschliessend Docker-basierte Workload-Cluster erstellen, verwalten, skalieren und aktualisieren.


In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
# Enable the experimental Cluster topology feature.
export CLUSTER_TOPOLOGY=true

# Initialize the management cluster
clusterctl init --infrastructure docker


### Cluster-Konfiguration erzeugen

In diesem Schritt wird festgelegt, welches virtuelle Maschinen-Image für die Cluster-Nodes verwendet wird und welche Kubernetes-Version installiert werden soll. Danach erzeugt Cluster API die vollständige Konfiguration für einen neuen Workload-Cluster mit einem Control-Plane-Node, zwei Worker Nodes und einem Load Balancer.

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
# The list of service CIDR, default ["10.128.0.0/12"]
export SERVICE_CIDR=["10.96.0.0/12"]
# The list of pod CIDR, default ["192.168.0.0/16"]
export POD_CIDR=["192.168.0.0/16"]
# The service domain, default "cluster.local"
export SERVICE_DOMAIN="k8s.test"

clusterctl generate cluster capi-docker --flavor development \
  --kubernetes-version v1.35.0 \
  --control-plane-machine-count=3 \
  --worker-machine-count=3 \
  --target-namespace=capi-docker \
  > capi-docker.yaml


Die erzeugte Datei `capi-docker.yaml` enthält alle Ressourcen, die Cluster API benötigt, um den Cluster auf der Docker-Umgebung bereitzustellen.

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl create namespace capi-docker
kubectl apply -f capi-docker.yaml

kubectl wait --for=condition=Available deployment --all --all-namespaces --timeout=180s

### Status der erstellten Ressourcen prüfen

Nach dem Erstellen des Workload-Clusters können die wichtigsten Cluster-API- und Docker-Ressourcen angezeigt werden:

Dabei ist insbesondere der Status der `KubeadmControlPlane` relevant. Die Control Plane muss vollständig bereit sein, bevor auf den neuen Workload-Cluster zugegriffen werden kann. Der Aufbau kann einige Zeit dauern, da Cluster API zuerst die virtuellen Maschinen erstellt, Kubernetes initialisiert und die Control-Plane-Nodes registriert.


In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl -n capi-docker get cluster,machinedeployment,machine
docker container ps

Sobald die `KubeadmControlPlane` bereit ist, kann die Kubeconfig des Workload-Clusters erzeugt werden:

Anschliessend kann mit dieser Kubeconfig auf den neu erstellten Cluster zugegriffen und geprüft werden, ob alle Systemkomponenten erfolgreich gestartet wurden:

In [ ]:
%%bash
kind get kubeconfig --name capi-docker >$HOME/data/.kube/capi-docker
kubectl --kubeconfig=$HOME/data/.kube/capi-docker get nodes -o wide 
kubectl --kubeconfig=$HOME/data/.kube/capi-docker get pods -A -o wide

### Calico als Overlay Netzwerk installieren

Ein Overlay Netzwerk muss im von Cluster API erstellten Workload-Cluster installiert werden, damit die Pods über ein funktionierendes CNI-Overlay-Netzwerk miteinander kommunizieren können.

In [ ]:
%%bash
kubectl --kubeconfig=$HOME/data/.kube/capi-docker apply -f https://raw.githubusercontent.com/projectcalico/calico/v3.26.1/manifests/calico.yaml

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl -n capi-docker get machinedeployment,machineset,machine,services
export KUBECONFIG=$HOME/data/.kube/capi-docker
kubectl get nodes
kubectl get pods,services -o wide -A

### Headlamp starten

Anschliessend wird Headlamp gestartet, um den Workload-Cluster über eine grafische Oberfläche zu überwachen und Kubernetes-Ressourcen einfacher zu verwalten.

**Achtung**: funktioniert nur auf dem Rechner wo clusterAPI läuft.

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-docker
kubectl apply -f https://raw.githubusercontent.com/headlamp-k8s/headlamp/main/kubernetes-headlamp.yaml
kubectl -n kube-system create serviceaccount headlamp-admin
kubectl create clusterrolebinding headlamp-admin --serviceaccount=kube-system:headlamp-admin --clusterrole=cluster-admin

kubectl patch svc headlamp \
  -n kube-system \
  --type='merge' \
  -p '{
    "spec": {
      "type": "NodePort",
      "ports": [
        {
          "port": 80,
          "targetPort": 4466,
          "nodePort": 30444
        }
      ]
    }
  }'
echo http://"$(kubectl get nodes -o jsonpath='{.items[0].status.addresses[?(@.type=="InternalIP")].address}{"\n"}')":30444
kubectl create token headlamp-admin -n kube-system --duration=48h

---

### Skalierung von Worker Nodes

Worker Nodes können über Machines hinzugefügt oder entfernt werden, um die Rechenkapazität des Clusters anzupassen. Die Skalierung erfolgt in der Regel über MachineSets oder MachineDeployments.

Beim Entfernen einer Machine wird der zugehörige Node kontrolliert geleert und anschliessend zusammen mit seiner Infrastruktur gelöscht.

**Links**
* [Skalierung](https://cluster-api.sigs.k8s.io/tasks/automated-machine-management/scaling)
* [Auto Skalierung](https://cluster-api.sigs.k8s.io/tasks/automated-machine-management/autoscaling)


In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl get --namespace capi-docker cluster,machine

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl patch cluster capi-docker \
  -n capi-docker \
  --type=merge \
  -p '{
    "spec": {
      "topology": {
        "workers": {
          "machineDeployments": [
            {
              "name": "md-0",
              "class": "default-worker",
              "replicas": 4
            }
          ]
        }
      }
    }
  }'

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl get --namespace capi-docker cluster,machine
export KUBECONFIG=$HOME/data/.kube/capi-docker
kubectl get nodes -o wide

- - -

### Updates von Kubernetes-Clustern

Cluster API ermöglicht kontrollierte Updates von Kubernetes-Clustern. Dabei wird zuerst die Control Plane und danach werden die Worker Nodes schrittweise auf die neue Version aktualisiert, damit der Cluster während des Updates möglichst verfügbar bleibt.

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl patch cluster capi-docker \
  -n capi-docker \
  --type=json \
  -p='[
    {
      "op":"replace",
      "path":"/spec/topology/version",
      "value":"v1.36.1"
    }
  ]'

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl get --namespace capi-docker cluster,machine
export KUBECONFIG=$HOME/data/.kube/capi-docker
kubectl get nodes -o wide

---

### Aufräumen


In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kind
kubectl delete --namespace capi-docker cluster capi-docker
kubectl delete namespace capi-docker
kind delete cluster --name capi-kind